<a href="https://colab.research.google.com/github/dcdlima/RAG_Tests/blob/main/RAG_Future_Forecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import chromadb
from chromadb.utils import embedding_functions
import pandas as pd
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------
# MÓDULO 1: Configuração e Expansão do Léxico (Keywords)
# ---------------------------------------------------------
KEYWORDS_FORESIGHT = [
    # Inglês
    "Tech Foresight", "Technology Forecast", "Future Events", "Roadmapping",
    "Future Wheel", "Horizon Scanning", "Trend Analysis", "Weak Signals",
    "Predicting", "Emerging Technologies", "Scenario Planning",
    # Português (BR)
    "Prospecção Tecnológica", "Previsão Tecnológica", "Eventos Futuros",
    "Mapeamento Tecnológico", "Roda do Futuro", "Monitoramento de Sinais Fracos",
    "Análise de Tendências", "Cenarização", "Tecnologias Emergentes", "Estudos do Futuro"
]

class TechForesightRAG:
    def __init__(self, collection_name="scientific_articles"):
        # Inicializa o cliente ChromaDB em memória para execução ágil
        self.chroma_client = chromadb.Client()

        # Utiliza um modelo de embedding multilíngue de alta performance
        # Raciocínio: Alinha perfeitamente termos técnicos em EN e PT-BR no mesmo espaço vetorial
        self.model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=self.model_name
        )

        # Cria ou obtém a coleção no banco vetorial
        self.collection = self.chroma_client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function,
            metadata={"hnsw:space": "cosine"} # Métrica de similaridade de cosseno
        )

    # ---------------------------------------------------------
    # MÓDULO 2: Ingestão e Fragmentação Semântica (Manual)
    # ---------------------------------------------------------
    def ingest_document(self, doc_id, text, metadata_base):
        """
        Recebe o texto completo de um artigo inserido manualmente,
        realiza o chunking semântico por quebra de sentenças/parágrafos
        e enriquece os metadados com base no léxico de prospecção.
        """
        # Chunking simples e eficiente por parágrafos para manter a unidade de tese
        chunks = [paragraph.strip() for paragraph in text.split("\n\n") if len(paragraph.strip()) > 50]

        documents = []
        metadatas = []
        ids = []

        for i, chunk in enumerate(chunks):
            # Análise lexical simples: verifica quais palavras-chave de futuro estão presentes no chunk
            found_keywords = [kw for kw in KEYWORDS_FORESIGHT if kw.lower() in chunk.lower()]
            has_future_signal = len(found_keywords) > 0

            # Construção dos metadados enriquecidos para filtragem precisa
            chunk_metadata = metadata_base.copy()
            chunk_metadata.update({
                "chunk_index": i,
                "has_future_signal": has_future_signal,
                "keywords_found": ", ".join(found_keywords) if has_future_signal else "None"
            })

            documents.append(chunk)
            metadatas.append(chunk_metadata)
            ids.append(f"{doc_id}_chunk_{i}")

        # Realiza a indexação no ChromaDB
        self.collection.add(
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )
        print(f"Sucesso: {len(documents)} fragmentos indexados para o documento {doc_id}.")

    # ---------------------------------------------------------
    # MÓDULO 4: Recuperação Híbrida/Semântica com Filtro
    # ---------------------------------------------------------
    def retrieve_context(self, query, n_results=3):
        """
        Busca os fragmentos mais relevantes aplicando filtro de metadados
        para priorizar trechos que contenham sinais explícitos de prospecção.
        """
        results = self.collection.query(
            query_texts=[query],
            n_results=n_results,
            # Raciocínio: Restringe ou prioriza chunks que possuem o sinal de futuro mapeado na ingestão
            where={"has_future_signal": True}
        )
        return results

    # ---------------------------------------------------------
    # MÓDULO 5: Geração de Texto de Prospecção (Simulação do LLM)
    # ---------------------------------------------------------
    def generate_future_insights(self, query):
        """
        Recupera o contexto e simula a chamada de um LLM estruturado
        retornando a correlação exata em formato de tabela.
        """
        retrieved = self.retrieve_context(query)

        source_texts = retrieved['documents'][0]
        metadatas = retrieved['metadatas'][0]

        output_data = []

        # Raciocínio do Prompt do LLM encapsulado na lógica de síntese:
        # O modelo deve agir estritamente sobre o contexto gerando análises preditivas limpas.
        for source_text, meta in zip(source_texts, metadatas):

            # Simulação de geração controlada do LLM (Grounding) baseada no trecho do artigo
            # Em produção, você substituiria esta lógica pela chamada de API (ex: OpenAI, Gemini, etc.)
            texto_gerado_llm = self._mock_llm_generation(source_text, meta)

            output_data.append({
                "Texto Fonte Utilizado": source_text,
                "Texto Gerado (Identificação do Futuro)": texto_gerado_llm
            })

        # Transforma o resultado em um DataFrame do Pandas para visualização em tabela
        df_output = pd.DataFrame(output_data)
        return df_output

    def _mock_llm_generation(self, source_text, metadata):
        """Simula a geração do LLM baseando-se estritamente no texto fonte."""
        # Exemplo simulado de extração conceitual do LLM
        return f"[ANÁLISE DE CENÁRIO ({metadata['keywords_found']})]: Com base nos dados de {metadata.get('year', 'N/A')}, mapeia-se uma rota tecnológica emergente onde o fragmento fonte aponta diretamente para a evolução do setor. O impacto previsto consolida o evento futuro extraído."

# ---------------------------------------------------------
# EXECUÇÃO DA ATIVIDADE (Modo de Uso)
# ---------------------------------------------------------
if __name__ == "__main__":
    # 1. Inicializa o Pipeline
    rag_pipeline = TechForesightRAG()

    # 2. Artigos inseridos manualmente (Exemplo de dados científicos simulados)
    artigo_1 = """
    A adoção de arquiteturas profundas de aprendizado de máquina transformou o mapeamento tecnológico.
    A prospecção tecnológica baseada em redes neurais indica que até 2030 teremos sistemas autônomos de decisão em refinarias.
    Este estudo de roadmapping aponta para uma convergência de sensoriamento IoT com modelos de linguagem.

    No passado, a manutenção era puramente reativa. Na década de 1990, os computadores apenas registravam falhas ocorridas.
    """

    artigo_2 = """
    We analyzed emerging technologies in energy storage systems. Our technology forecast suggests
    that solid-state batteries will reach commercial maturity by 2028, creating new weak signals
    in the automotive supply chain. This future wheel exercise evaluates secondary impacts on lithium mining.
    """

    # 3. Ingestão manual com metadados básicos
    rag_pipeline.ingest_document(doc_id="Artigo_O&G_2026", text=artigo_1, metadata_base={"year": 2026, "author": "Silva et al."})
    rag_pipeline.ingest_document(doc_id="Artigo_Energy_2025", text=artigo_2, metadata_base={"year": 2025, "author": "Smith et al."})

    # 4. Execução da busca e geração do relatório preditivo
    query_busca = "Quais são as tecnologias emergentes e previsões de mercado citadas?"
    tabela_resultados = rag_pipeline.generate_future_insights(query_busca)

    # Exibe a tabela final formatada
    pd.set_option('display.max_colwidth', None)
    print("\n--- TABELA DE SAÍDA DO PIPELINE RAG ---")
    print(tabela_resultados.to_markdown(index=False))

ModuleNotFoundError: No module named 'chromadb'